# MedSAM2 Ventricle Segmentation — Fine-Tuning Pipeline

**Author:** Matheus Machado Rech  
**Project:** HydroMorph RN — Hydrocephalus Morphometrics App  
**License:** Research use only — not for clinical diagnosis

---

This notebook fine-tunes [MedSAM2](https://github.com/bowang-lab/MedSAM2) specifically for brain ventricle segmentation from head CT scans. MedSAM2 treats 3D medical volumes as "video" sequences where each axial slice is a frame, making it well-suited for volumetric segmentation.

**Fine-tuning strategy:**
- Freeze the image encoder (Hiera backbone) to preserve learned visual features
- Train only the mask decoder and prompt encoder — faster, less GPU memory
- Use bounding box prompts derived from ground truth ventricle masks
- Mixed precision training (AMP) for memory efficiency on T4 GPU

**Training data format (MedSAM2 NPZ):**
- `imgs`: (Z, Y, X) uint8 array [0, 255] — brain-windowed CT slices
- `gts`: (Z, Y, X) binary ground truth masks — ventricle segmentation
- `spacing`: (3,) float array — voxel spacing in mm

**Prerequisites:**
- Google Colab with T4 GPU runtime (Runtime > Change runtime type > T4 GPU)
- ~8 GB GPU memory (T4 provides 15 GB)

**Estimated training time:** ~20-30 minutes for 10 synthetic volumes, 50 epochs on T4.

---

### Table of Contents
1. [Environment Setup](#environment-setup)
2. [Data Preparation](#data-preparation)
3. [Train/Val Split](#train-val-split)
4. [Fine-Tuning Setup](#fine-tuning-setup)
5. [Training Loop](#training-loop)
6. [Validation and Metrics](#validation-and-metrics)
7. [Export for Deployment](#export-for-deployment)
8. [Comparison: Base vs Fine-Tuned](#comparison)
9. [Next Steps](#next-steps)

In [ ]:
# =============================================================================
# 1. Environment Setup
# =============================================================================
# Clone MedSAM2 repository (pinned to a known-good commit) and install deps.
# Download the base checkpoint and verify its SHA-256 integrity.
# Estimated time: ~3-5 minutes on Colab.

import os
import sys
import hashlib

# Verify GPU availability
import subprocess
gpu_info = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total,memory.free', '--format=csv,noheader'],
    capture_output=True, text=True
)
print(f"GPU: {gpu_info.stdout.strip()}")
assert 'T4' in gpu_info.stdout or 'GPU' in gpu_info.stdout, \
    "No GPU detected. Go to Runtime > Change runtime type > T4 GPU"

# Clone MedSAM2 at a pinned commit
MEDSAM2_COMMIT = '332f30d420f1d1b08e2a79b3ae6a602458808383'
if not os.path.exists('/content/MedSAM2'):
    !git clone https://github.com/bowang-lab/MedSAM2.git /content/MedSAM2
    !git -C /content/MedSAM2 checkout {MEDSAM2_COMMIT}
    print(f"MedSAM2 cloned and checked out at {MEDSAM2_COMMIT}")
else:
    # Verify the existing clone is at the expected commit
    actual = subprocess.run(['git', '-C', '/content/MedSAM2', 'rev-parse', '--short', 'HEAD'],
                             capture_output=True, text=True).stdout.strip()
    assert actual == MEDSAM2_COMMIT, \
        f"Unexpected MedSAM2 commit {actual!r} — expected {MEDSAM2_COMMIT!r}. "\
        "Re-run after deleting /content/MedSAM2."
    print(f"MedSAM2 already cloned at {actual}")

%cd /content/MedSAM2
!git fetch --depth 1 origin 332f30d420f1d1b08e2a79b3ae6a602458808383
!git checkout 332f30d420f1d1b08e2a79b3ae6a602458808383

# Install MedSAM2 in editable mode
!pip install -e . -q

# Install additional dependencies
!pip install nibabel scikit-image scipy tqdm -q

# Download MedSAM2 checkpoint (~149 MB) from a pinned revision
MEDSAM2_HF_REVISION = 'e4a6f35edd7e091619cbc0750f462f1574e23955'
EXPECTED_CHECKPOINT_SHA256 = 'dcd946a4d934f553236866fc7e8af77f7e931430e9c044f4ac9d6a723630a870'

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

os.makedirs('checkpoints', exist_ok=True)
checkpoint_path = 'checkpoints/MedSAM2_2411.pt'
if not os.path.exists(checkpoint_path):
    !wget -q --show-progress -P checkpoints \
        https://huggingface.co/wanglab/MedSAM2/resolve/e4a6f35edd7e091619cbc0750f462f1574e23955/MedSAM2_2411.pt
    print(f"Checkpoint downloaded: {os.path.getsize(checkpoint_path) / 1e6:.1f} MB")
else:
    print(f"Checkpoint already exists: {os.path.getsize(checkpoint_path) / 1e6:.1f} MB")

# ── Integrity check ────────────────────────────────────────────────────────
# Always verify the checkpoint before loading it to detect corruption or
# tampering.  Set CHECKPOINT_SHA256 above to the authoritative hash.
if CHECKPOINT_SHA256 is not None:
    sha256 = hashlib.sha256()
    with open(checkpoint_path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            sha256.update(chunk)
    actual_hash = sha256.hexdigest()
    assert actual_hash == CHECKPOINT_SHA256, \
        f"Checkpoint SHA-256 mismatch!\n  expected: {CHECKPOINT_SHA256}\n  actual:   {actual_hash}\n"\
        "Do NOT load this file — re-download from HuggingFace and update CHECKPOINT_SHA256."
    print(f"Checkpoint integrity verified: {actual_hash[:16]}...")
else:
    import warnings
    warnings.warn(
        "CHECKPOINT_SHA256 is not set. Set it to the known-good SHA-256 of "
        "MedSAM2_2411.pt before running in a production or clinical environment.",
        stacklevel=2,
    )
actual_sha256 = sha256_file(checkpoint_path)
if actual_sha256 != EXPECTED_CHECKPOINT_SHA256:
    raise RuntimeError(
        f"Checkpoint SHA256 mismatch: {actual_sha256} != {EXPECTED_CHECKPOINT_SHA256}"
    )
print(f"Checkpoint SHA256 verified: {actual_sha256}")

# Verify imports
import torch
import numpy as np
import nibabel as nib
from PIL import Image
import matplotlib.pyplot as plt
from skimage import measure
from scipy.ndimage import binary_fill_holes, binary_erosion, binary_dilation
from tqdm.auto import tqdm

print(f"\nPyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"GPU memory: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")
print(f"\nEnvironment ready.")


<a id="data-preparation"></a>
## 2. Data Preparation

MedSAM2 expects training data as **NPZ files** with the following keys:
- `imgs`: (Z, Y, X) uint8 array with values in [0, 255] — brain-windowed CT
- `gts`: (Z, Y, X) binary ground truth masks (0 = background, 1 = ventricle)
- `spacing`: (3,) float array — voxel spacing in mm (z, y, x)

**Brain windowing parameters:**
- Window Level (WL): 40 HU
- Window Width (WW): 80 HU
- Clipping range: [0, 80] HU, normalized to [0, 255]

This section provides:
1. `nifti_to_npz()` — converts NIfTI CT + mask pairs to MedSAM2 NPZ format
2. Synthetic training data generation (10 volumes with varying ventricle sizes)
3. Visualization of a sample NPZ volume

For production training, replace synthetic data with real ventricle-labeled CT scans from TotalSegmentator or institutional datasets.

In [ ]:
# =============================================================================
# 2. Data Preparation — NIfTI to NPZ Conversion + Synthetic Data
# =============================================================================
import numpy as np
import nibabel as nib
import os
import matplotlib.pyplot as plt

os.makedirs('data/npz', exist_ok=True)


def nifti_to_npz(ct_path, mask_path, output_path, wl=40, ww=80):
    """
    Convert a NIfTI CT volume + ventricle mask to MedSAM2 NPZ format.

    Preprocessing steps:
    1. Load NIfTI files and extract voxel data
    2. Reorient both to RAS+ standard orientation
    3. Apply brain window: clip HU to [WL-WW/2, WL+WW/2], normalize to [0, 255]
    4. Transpose to (Z, Y, X) for MedSAM2's video format
    5. Save as NPZ with imgs, gts, spacing keys

    Parameters
    ----------
    ct_path : str
        Path to brain CT NIfTI file (HU values).
    mask_path : str
        Path to binary ventricle mask NIfTI file.
    output_path : str
        Output NPZ file path.
    wl : int
        Window level in HU (default: 40).
    ww : int
        Window width in HU (default: 80).

    Returns
    -------
    dict
        Statistics about the converted volume.
    """
    # Load NIfTI
    ct_nii = nib.load(ct_path)
    ct_vol = ct_nii.get_fdata().astype(np.float32)
    mask_vol = nib.load(mask_path).get_fdata().astype(np.uint8)
    spacing = np.array(ct_nii.header.get_zooms()[:3], dtype=np.float32)

    # Reorient to RAS+
    ornt = nib.orientations.io_orientation(ct_nii.affine)
    ras_ornt = nib.orientations.axcodes2ornt(('R', 'A', 'S'))
    if not np.array_equal(ornt, ras_ornt):
        transform = nib.orientations.ornt_transform(ornt, ras_ornt)
        ct_vol = nib.orientations.apply_orientation(ct_vol, transform)
        mask_vol = nib.orientations.apply_orientation(mask_vol, transform)

    # Brain window: clip to [lower, upper], normalize to [0, 255]
    lower = wl - ww // 2  # 0
    upper = wl + ww // 2  # 80
    ct_windowed = np.clip(ct_vol, lower, upper)
    imgs = ((ct_windowed - lower) / (upper - lower) * 255).astype(np.uint8)

    # Binarize mask
    gts = (mask_vol > 0).astype(np.uint8)

    # Transpose to (Z, Y, X) for MedSAM2 video format
    imgs = np.transpose(imgs, (2, 1, 0))  # (Z, Y, X)
    gts = np.transpose(gts, (2, 1, 0))
    spacing_zyx = np.array([spacing[2], spacing[1], spacing[0]], dtype=np.float32)

    # Save NPZ
    np.savez_compressed(output_path, imgs=imgs, gts=gts, spacing=spacing_zyx)

    stats = {
        'shape': imgs.shape,
        'spacing': spacing_zyx.tolist(),
        'ventricle_voxels': int(gts.sum()),
        'ventricle_volume_ml': float(gts.sum() * np.prod(spacing_zyx) / 1000.0),
        'slices_with_ventricle': int(np.sum(gts.sum(axis=(1, 2)) > 0)),
        'file_size_mb': os.path.getsize(output_path) / 1e6,
    }
    return stats


def create_synthetic_training_data(n_cases=10, output_dir='data/npz'):
    """
    Create synthetic brain CT volumes with ventricle masks for training.

    Each volume features:
    - Ellipsoidal skull shell (HU ~700)
    - Brain parenchyma with noise (HU ~30)
    - Bilateral lateral ventricles (HU ~10) with varying sizes
    - Third ventricle (small, midline)
    - Random anatomical variation (center shift, ventricle scale)

    Ventricle scale ranges from 0.7 (normal) to 1.5 (severely dilated)
    to simulate the spectrum from healthy to NPH.

    Parameters
    ----------
    n_cases : int
        Number of synthetic volumes to generate.
    output_dir : str
        Output directory for NPZ files.

    Returns
    -------
    list of str
        Paths to generated NPZ files.
    """
    os.makedirs(output_dir, exist_ok=True)
    npz_paths = []

    for case_id in range(n_cases):
        rng = np.random.default_rng(seed=case_id)

        shape = (256, 256, 128)
        volume = np.full(shape, -1000, dtype=np.int16)  # Air background

        # Random anatomical variation
        cx = shape[0] // 2 + rng.integers(-5, 6)
        cy = shape[1] // 2 + rng.integers(-5, 6)
        cz = shape[2] // 2

        # Ventricle scale: 0.7 (normal) to 1.5 (severely dilated NPH)
        vent_scale = 0.7 + (case_id / (n_cases - 1)) * 0.8 if n_cases > 1 else 1.0

        # Coordinate grids
        x = np.arange(shape[0])
        y = np.arange(shape[1])
        z = np.arange(shape[2])
        X, Y, Z = np.meshgrid(x, y, z, indexing='ij')

        # Brain ellipsoid
        dx_brain = (X - cx) / 90.0
        dy_brain = (Y - cy) / 100.0
        dz_brain = (Z - cz) / 44.0
        r_brain = dx_brain**2 + dy_brain**2 + dz_brain**2

        # Skull ellipsoid
        dx_skull = (X - cx) / 95.0
        dy_skull = (Y - cy) / 105.0
        dz_skull = (Z - cz) / 47.0
        r_skull = dx_skull**2 + dy_skull**2 + dz_skull**2

        # Skull shell (HU ~700)
        skull_mask = (r_skull <= 1.1) & (r_skull >= 0.92)
        volume[skull_mask] = 700

        # Brain parenchyma (HU ~30) with noise
        brain_mask = r_brain <= 1.0
        volume[brain_mask & ~skull_mask] = 30
        noise = rng.normal(0, 5, shape).astype(np.int16)
        volume[brain_mask & ~skull_mask] += noise[brain_mask & ~skull_mask]

        # Left lateral ventricle (HU ~10)
        lv_rx, lv_ry, lv_rz = 13.0 * vent_scale, 22.0 * vent_scale, 19.0 * vent_scale
        dx_lv = (X - (cx - 15)) / lv_rx
        dy_lv = (Y - cy) / lv_ry
        dz_lv = (Z - cz) / lv_rz
        left_vent = (dx_lv**2 + dy_lv**2 + dz_lv**2) <= 1.0
        volume[left_vent] = 10

        # Right lateral ventricle (HU ~10)
        dx_rv = (X - (cx + 15)) / lv_rx
        dy_rv = (Y - cy) / lv_ry
        dz_rv = (Z - cz) / lv_rz
        right_vent = (dx_rv**2 + dy_rv**2 + dz_rv**2) <= 1.0
        volume[right_vent] = 10

        # Third ventricle (small, midline)
        tv_scale = 1.0 + (vent_scale - 1.0) * 0.5  # Third vent dilates less
        dx_3v = (X - cx) / (3.0 * tv_scale)
        dy_3v = (Y - cy) / (8.0 * tv_scale)
        dz_3v = (Z - cz) / (10.0 * tv_scale)
        third_vent = (dx_3v**2 + dy_3v**2 + dz_3v**2) <= 1.0
        volume[third_vent] = 10

        # Ground truth mask
        gt_mask = np.zeros(shape, dtype=np.uint8)
        gt_mask[left_vent | right_vent | third_vent] = 1

        # Brain window: clip [0, 80], normalize to [0, 255]
        ct_windowed = np.clip(volume.astype(np.float32), 0, 80)
        imgs = ((ct_windowed / 80.0) * 255).astype(np.uint8)

        # Transpose to (Z, Y, X) for MedSAM2
        spacing = np.array([1.5, 1.0, 1.0], dtype=np.float32)  # Z, Y, X
        imgs_zyx = np.transpose(imgs, (2, 1, 0))
        gts_zyx = np.transpose(gt_mask, (2, 1, 0))

        # Save NPZ
        npz_path = os.path.join(output_dir, f'ventricle_case_{case_id:04d}.npz')
        np.savez_compressed(npz_path, imgs=imgs_zyx, gts=gts_zyx, spacing=spacing)
        npz_paths.append(npz_path)

        vent_vol_ml = gt_mask.sum() * np.prod(spacing) / 1000.0
        print(f"  Case {case_id:2d}: scale={vent_scale:.2f}, "
              f"vent_vol={vent_vol_ml:.1f} mL, "
              f"slices_w_vent={np.sum(gts_zyx.sum(axis=(1,2)) > 0)}, "
              f"size={os.path.getsize(npz_path)/1e6:.1f} MB")

    return npz_paths


# --- Generate synthetic training data ---
print("Generating 10 synthetic brain CT volumes with ventricle masks...")
print("Ventricle scale ranges from 0.70 (normal) to 1.50 (severe NPH)\n")
npz_paths = create_synthetic_training_data(n_cases=10)
print(f"\nGenerated {len(npz_paths)} NPZ files in data/npz/")

# --- Visualize a sample NPZ ---
sample = np.load(npz_paths[5])  # Mid-range ventricle size
sample_imgs = sample['imgs']
sample_gts = sample['gts']
sample_spacing = sample['spacing']

print(f"\nSample NPZ (case 5):")
print(f"  imgs shape: {sample_imgs.shape}, dtype: {sample_imgs.dtype}")
print(f"  gts shape:  {sample_gts.shape}, dtype: {sample_gts.dtype}")
print(f"  spacing:    {sample_spacing} mm")
print(f"  pixel range: [{sample_imgs.min()}, {sample_imgs.max()}]")

# Plot 4 representative slices
n_slices = sample_imgs.shape[0]
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i, frac in enumerate([0.35, 0.42, 0.50, 0.58]):
    z = int(n_slices * frac)
    axes[0, i].imshow(sample_imgs[z], cmap='gray')
    axes[0, i].set_title(f'Slice {z} (brain window)', fontsize=10)
    axes[0, i].axis('off')

    # Overlay ground truth mask
    axes[1, i].imshow(sample_imgs[z], cmap='gray')
    mask_rgba = np.zeros((*sample_gts[z].shape, 4))
    mask_rgba[:, :, 0] = 1.0  # Red channel
    mask_rgba[:, :, 1] = 0.8  # Green (yellow-ish)
    mask_rgba[:, :, 3] = sample_gts[z].astype(float) * 0.5  # Alpha
    axes[1, i].imshow(mask_rgba)
    n_px = sample_gts[z].sum()
    axes[1, i].set_title(f'GT overlay ({n_px} px)', fontsize=10)
    axes[1, i].axis('off')

plt.suptitle('Sample NPZ -- Brain-Windowed CT + Ventricle Ground Truth', fontsize=14)
plt.tight_layout()
plt.show()

<a id="train-val-split"></a>
## 3. Train/Val Split

Split the NPZ files into training (80%) and validation (20%) sets.

The split is done by **case** (not by slice) to prevent data leakage. Each NPZ file represents one complete 3D volume, so all slices from the same patient stay in the same split.

In [ ]:
# =============================================================================
# 3. Train/Val Split
# =============================================================================
import shutil
import random
from pathlib import Path

# Create train/val directories
for split in ['train', 'val']:
    os.makedirs(f'data/{split}', exist_ok=True)

# Get all NPZ files
all_npz = sorted(Path('data/npz').glob('ventricle_case_*.npz'))
print(f"Total NPZ files: {len(all_npz)}")

# Shuffle and split 80/20
random.seed(42)
indices = list(range(len(all_npz)))
random.shuffle(indices)

n_val = max(1, int(len(all_npz) * 0.2))
val_indices = set(indices[:n_val])
train_indices = set(indices[n_val:])

train_files = []
val_files = []

for i, npz_path in enumerate(all_npz):
    if i in val_indices:
        dest = f'data/val/{npz_path.name}'
        shutil.copy2(str(npz_path), dest)
        val_files.append(dest)
    else:
        dest = f'data/train/{npz_path.name}'
        shutil.copy2(str(npz_path), dest)
        train_files.append(dest)

print(f"\nTrain set: {len(train_files)} volumes")
for f in train_files:
    data = np.load(f)
    n_vent = data['gts'].sum()
    vol_ml = n_vent * np.prod(data['spacing']) / 1000.0
    print(f"  {Path(f).name}: shape={data['imgs'].shape}, vent_vol={vol_ml:.1f} mL")

print(f"\nVal set: {len(val_files)} volumes")
for f in val_files:
    data = np.load(f)
    n_vent = data['gts'].sum()
    vol_ml = n_vent * np.prod(data['spacing']) / 1000.0
    print(f"  {Path(f).name}: shape={data['imgs'].shape}, vent_vol={vol_ml:.1f} mL")

<a id="fine-tuning-setup"></a>
## 4. MedSAM2 Fine-Tuning Setup

### Strategy: Freeze Encoder, Train Decoder

MedSAM2 has three main components:
1. **Image encoder (Hiera)** -- extracts visual features from each slice (~80% of parameters)
2. **Prompt encoder** -- encodes bounding box / point prompts into embeddings
3. **Mask decoder** -- generates segmentation masks from image + prompt features

**We freeze the image encoder** and only train the mask decoder + prompt encoder. This:
- Reduces trainable parameters by ~80%
- Prevents overfitting on small datasets
- Requires less GPU memory (no encoder gradients)
- Converges faster (~50 epochs vs 200+)

### Bounding Box Prompt Generation

During training, we derive bounding box prompts from the ground truth mask on each key slice:
1. Find the tightest bounding box around the ventricle mask
2. Add 10-20% random margin (data augmentation)
3. Scale to 512x512 coordinate space (MedSAM2's internal resolution)

### Loss Function

Combined **Dice loss + Binary Cross-Entropy (BCE)** loss:
- Dice loss handles class imbalance (ventricles are ~2-5% of brain volume)
- BCE provides stable gradients for fine-grained boundary learning
- Weight: 0.5 * Dice + 0.5 * BCE

In [ ]:
# =============================================================================
# 4. Model Setup and Training Configuration
# =============================================================================
import torch
import torch.nn as nn
import torch.nn.functional as F
from sam2.build_sam import build_sam2_video_predictor_npz
from PIL import Image

torch.set_float32_matmul_precision('high')

# --- Load MedSAM2 ---
checkpoint = './checkpoints/MedSAM2_2411.pt'
model_cfg = 'configs/sam2.1_hiera_t512.yaml'

print("Loading MedSAM2 model...")
predictor = build_sam2_video_predictor_npz(model_cfg, checkpoint)
model = predictor.model  # Access the underlying SAM2 model
model.train()
print(f"Model loaded.")

# --- Freeze image encoder ---
frozen_params = 0
trainable_params = 0

for name, param in model.named_parameters():
    if 'image_encoder' in name:
        param.requires_grad = False
        frozen_params += param.numel()
    else:
        param.requires_grad = True
        trainable_params += param.numel()

total_params = frozen_params + trainable_params
print(f"\nParameter Summary:")
print(f"  Total parameters:     {total_params:>12,}")
print(f"  Frozen (encoder):     {frozen_params:>12,} ({100*frozen_params/total_params:.1f}%)")
print(f"  Trainable (decoder):  {trainable_params:>12,} ({100*trainable_params/total_params:.1f}%)")

# --- Loss function: Dice + BCE ---
class DiceBCELoss(nn.Module):
    """
    Combined Dice loss + Binary Cross-Entropy loss.

    Dice loss handles class imbalance (ventricles are small relative to
    the full brain volume). BCE provides stable gradients for fine-grained
    boundary learning.

    Parameters
    ----------
    dice_weight : float
        Weight for Dice loss component (default: 0.5).
    bce_weight : float
        Weight for BCE loss component (default: 0.5).
    smooth : float
        Smoothing factor for Dice computation (default: 1.0).
    """
    def __init__(self, dice_weight=0.5, bce_weight=0.5, smooth=1.0):
        super().__init__()
        self.dice_weight = dice_weight
        self.bce_weight = bce_weight
        self.smooth = smooth

    def forward(self, logits, targets):
        """
        Compute combined loss.

        Parameters
        ----------
        logits : torch.Tensor
            Raw model output (before sigmoid).
        targets : torch.Tensor
            Binary ground truth mask.
        """
        probs = torch.sigmoid(logits)

        # Flatten for computation
        probs_flat = probs.view(-1)
        targets_flat = targets.view(-1).float()

        # Dice loss
        intersection = (probs_flat * targets_flat).sum()
        dice = (2. * intersection + self.smooth) / (
            probs_flat.sum() + targets_flat.sum() + self.smooth
        )
        dice_loss = 1.0 - dice

        # BCE loss
        bce_loss = F.binary_cross_entropy_with_logits(logits.view(-1), targets_flat)

        return self.dice_weight * dice_loss + self.bce_weight * bce_loss


# --- Training configuration ---
config = {
    'lr': 1e-4,
    'weight_decay': 1e-4,
    'epochs': 50,
    'image_size': 512,
    'dice_weight': 0.5,
    'bce_weight': 0.5,
    'bbox_margin_range': (0.10, 0.20),  # Random margin augmentation
    'min_ventricle_area': 50,  # Minimum pixels to count as valid slice
    'grad_clip': 1.0,
    'checkpoint_dir': 'checkpoints/finetuned',
}

os.makedirs(config['checkpoint_dir'], exist_ok=True)

# --- Optimizer: AdamW (only unfrozen parameters) ---
criterion = DiceBCELoss(
    dice_weight=config['dice_weight'],
    bce_weight=config['bce_weight']
)
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=config['lr'],
    weight_decay=config['weight_decay']
)
scaler = torch.amp.GradScaler('cuda')  # For mixed precision training

print(f"\nTraining Configuration:")
for k, v in config.items():
    print(f"  {k}: {v}")
print(f"\nOptimizer: AdamW (lr={config['lr']}, wd={config['weight_decay']})")
print(f"Loss: DiceBCE (dice_w={config['dice_weight']}, bce_w={config['bce_weight']})")
print(f"Mixed precision: enabled (AMP + GradScaler)")

In [ ]:
# =============================================================================
# 5. Training Loop
# =============================================================================
import time
from tqdm.auto import tqdm
from skimage import measure


def resize_grayscale_to_rgb(array, image_size=512):
    """
    Resize (D, H, W) grayscale volume to (D, 3, image_size, image_size) RGB.

    MedSAM2 expects RGB input at 512x512 resolution.
    Each grayscale slice is replicated across 3 channels.
    """
    d, h, w = array.shape
    resized = np.zeros((d, 3, image_size, image_size), dtype=np.float32)
    for i in range(d):
        img_pil = Image.fromarray(array[i].astype(np.uint8))
        img_rgb = img_pil.convert('RGB')
        img_resized = img_rgb.resize((image_size, image_size), Image.BILINEAR)
        resized[i] = np.array(img_resized).transpose(2, 0, 1)
    return resized


def get_bbox_from_mask(mask_2d, margin_range=(0.10, 0.20)):
    """
    Compute bounding box from a 2D binary mask with random margin.

    Parameters
    ----------
    mask_2d : np.ndarray
        Binary mask (H, W).
    margin_range : tuple
        (min_margin, max_margin) as fraction of box size.

    Returns
    -------
    np.ndarray
        Bounding box [x1, y1, x2, y2] in pixel coordinates.
    """
    rows = np.any(mask_2d, axis=1)
    cols = np.any(mask_2d, axis=0)
    rmin, rmax = np.where(rows)[0][[0, -1]]
    cmin, cmax = np.where(cols)[0][[0, -1]]

    H, W = mask_2d.shape
    margin = np.random.uniform(*margin_range)
    margin_y = max(int((rmax - rmin) * margin), 3)
    margin_x = max(int((cmax - cmin) * margin), 3)

    return np.array([
        max(0, cmin - margin_x),
        max(0, rmin - margin_y),
        min(W - 1, cmax + margin_x),
        min(H - 1, rmax + margin_y)
    ], dtype=np.float32)


def compute_dice(pred, gt):
    """Compute Dice coefficient between two binary masks."""
    pred = pred.astype(bool)
    gt = gt.astype(bool)
    if pred.sum() == 0 and gt.sum() == 0:
        return 1.0
    intersection = np.logical_and(pred, gt).sum()
    return 2.0 * intersection / (pred.sum() + gt.sum() + 1e-8)


def get_largest_cc(segmentation):
    """Keep only the largest connected component in a 3D binary mask."""
    labels = measure.label(segmentation, connectivity=2)
    if labels.max() == 0:
        return segmentation
    component_sizes = np.bincount(labels.flat)
    component_sizes[0] = 0  # Ignore background
    largest_label = np.argmax(component_sizes)
    return (labels == largest_label).astype(np.uint8)


def train_one_epoch(predictor, train_files, config, criterion, optimizer, scaler):
    """
    Train MedSAM2 for one epoch over all training NPZ files.

    For each volume:
    1. Pick a random key slice with ventricle presence
    2. Compute bounding box from GT with random margin augmentation
    3. Resize + normalize for MedSAM2 input
    4. Run forward pass (init_state, add box, propagate)
    5. Compute loss on predicted vs GT masks
    6. Backpropagate through unfrozen parameters

    Returns
    -------
    float
        Mean loss over the epoch.
    """
    model = predictor.model
    model.train()
    epoch_losses = []
    image_size = config['image_size']

    # ImageNet normalization constants
    img_mean = torch.tensor([0.485, 0.456, 0.406], dtype=torch.float32)[:, None, None].cuda()
    img_std = torch.tensor([0.229, 0.224, 0.225], dtype=torch.float32)[:, None, None].cuda()

    for npz_path in train_files:
        data = np.load(npz_path)
        imgs = data['imgs']  # (Z, Y, X) uint8
        gts = data['gts']    # (Z, Y, X) binary

        video_height, video_width = imgs.shape[1], imgs.shape[2]

        # Find slices with ventricle presence
        slice_areas = gts.sum(axis=(1, 2))
        valid_slices = np.where(slice_areas >= config['min_ventricle_area'])[0]
        if len(valid_slices) == 0:
            continue

        # Pick random key slice
        key_slice = int(np.random.choice(valid_slices))

        # Compute bounding box from GT on key slice
        gt_key = gts[key_slice]
        bbox = get_bbox_from_mask(gt_key, margin_range=config['bbox_margin_range'])

        # Resize volume for MedSAM2 input
        img_resized = resize_grayscale_to_rgb(imgs, image_size=image_size)
        img_resized = img_resized / 255.0
        img_tensor = torch.from_numpy(img_resized).cuda().float()
        img_tensor -= img_mean
        img_tensor /= img_std

        # Scale bbox to 512x512 space
        scale_x = float(image_size) / video_width
        scale_y = float(image_size) / video_height
        bbox_scaled = np.array([
            bbox[0] * scale_x, bbox[1] * scale_y,
            bbox[2] * scale_x, bbox[3] * scale_y,
        ])

        # Resize GT masks to model output resolution
        gts_resized = np.zeros((gts.shape[0], image_size, image_size), dtype=np.float32)
        for z in range(gts.shape[0]):
            gt_pil = Image.fromarray(gts[z].astype(np.uint8) * 255)
            gt_resized = gt_pil.resize((image_size, image_size), Image.NEAREST)
            gts_resized[z] = np.array(gt_resized).astype(np.float32) / 255.0
        gts_tensor = torch.from_numpy(gts_resized).cuda().float()

        optimizer.zero_grad()

        try:
            with torch.amp.autocast('cuda', dtype=torch.bfloat16):
                # Forward propagation
                inference_state = predictor.init_state(img_tensor, video_height, video_width)
                _, _, out_mask_logits = predictor.add_new_points_or_box(
                    inference_state=inference_state,
                    frame_idx=key_slice,
                    obj_id=1,
                    box=bbox_scaled,
                )

                # Propagate and collect losses
                total_loss = torch.tensor(0.0, device='cuda')
                n_frames = 0

                for out_frame_idx, _, out_logits in predictor.propagate_in_video(inference_state):
                    logit = out_logits[0]  # (1, H, W)
                    gt_frame = gts_tensor[out_frame_idx].unsqueeze(0)  # (1, H, W)

                    # Resize logit to match GT if needed
                    if logit.shape[-2:] != gt_frame.shape[-2:]:
                        logit = F.interpolate(
                            logit.unsqueeze(0),
                            size=gt_frame.shape[-2:],
                            mode='bilinear',
                            align_corners=False
                        ).squeeze(0)

                    total_loss += criterion(logit, gt_frame)
                    n_frames += 1

                predictor.reset_state(inference_state)

                # Backward propagation (reverse direction)
                inference_state = predictor.init_state(img_tensor, video_height, video_width)
                _, _, _ = predictor.add_new_points_or_box(
                    inference_state=inference_state,
                    frame_idx=key_slice,
                    obj_id=1,
                    box=bbox_scaled,
                )
                for out_frame_idx, _, out_logits in predictor.propagate_in_video(
                    inference_state, reverse=True
                ):
                    logit = out_logits[0]
                    gt_frame = gts_tensor[out_frame_idx].unsqueeze(0)
                    if logit.shape[-2:] != gt_frame.shape[-2:]:
                        logit = F.interpolate(
                            logit.unsqueeze(0),
                            size=gt_frame.shape[-2:],
                            mode='bilinear',
                            align_corners=False
                        ).squeeze(0)
                    total_loss += criterion(logit, gt_frame)
                    n_frames += 1

                predictor.reset_state(inference_state)

                if n_frames > 0:
                    mean_loss = total_loss / n_frames
                else:
                    continue

            # Backward pass with gradient scaling
            scaler.scale(mean_loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(
                filter(lambda p: p.requires_grad, model.parameters()),
                config['grad_clip']
            )
            scaler.step(optimizer)
            scaler.update()

            epoch_losses.append(mean_loss.item())

        except RuntimeError as e:
            print(f"  WARNING: Error processing {Path(npz_path).name}: {e}")
            predictor.reset_state(inference_state)
            optimizer.zero_grad()
            torch.cuda.empty_cache()
            continue

    return np.mean(epoch_losses) if epoch_losses else float('inf')


@torch.inference_mode()
def validate(predictor, val_files, config):
    """
    Run validation on all val volumes and compute mean Dice score.

    Uses the slice with maximum ventricle area as key slice,
    derives bounding box from GT (fixed 15% margin, no randomness),
    and runs full forward + backward propagation.

    Returns
    -------
    float
        Mean Dice score across all validation volumes.
    """
    model = predictor.model
    model.eval()
    dice_scores = []
    image_size = config['image_size']

    img_mean = torch.tensor([0.485, 0.456, 0.406], dtype=torch.float32)[:, None, None].cuda()
    img_std = torch.tensor([0.229, 0.224, 0.225], dtype=torch.float32)[:, None, None].cuda()

    for npz_path in val_files:
        data = np.load(npz_path)
        imgs = data['imgs']
        gts = data['gts']

        video_height, video_width = imgs.shape[1], imgs.shape[2]

        # Find key slice (max ventricle area)
        slice_areas = gts.sum(axis=(1, 2))
        key_slice = int(np.argmax(slice_areas))
        if slice_areas[key_slice] < config['min_ventricle_area']:
            continue

        # Fixed margin bbox from GT
        bbox = get_bbox_from_mask(gts[key_slice], margin_range=(0.15, 0.15))

        # Preprocess
        img_resized = resize_grayscale_to_rgb(imgs, image_size=image_size)
        img_resized = img_resized / 255.0
        img_tensor = torch.from_numpy(img_resized).cuda().float()
        img_tensor -= img_mean
        img_tensor /= img_std

        scale_x = float(image_size) / video_width
        scale_y = float(image_size) / video_height
        bbox_scaled = np.array([
            bbox[0] * scale_x, bbox[1] * scale_y,
            bbox[2] * scale_x, bbox[3] * scale_y,
        ])

        # Run inference
        segs_3D = np.zeros(imgs.shape, dtype=np.uint8)

        with torch.autocast('cuda', dtype=torch.bfloat16):
            # Forward
            inference_state = predictor.init_state(img_tensor, video_height, video_width)
            _, _, _ = predictor.add_new_points_or_box(
                inference_state=inference_state,
                frame_idx=key_slice, obj_id=1, box=bbox_scaled,
            )
            for out_idx, _, out_logits in predictor.propagate_in_video(inference_state):
                mask = (out_logits[0] > 0.0).cpu().numpy()[0]
                segs_3D[out_idx][mask] = 1
            predictor.reset_state(inference_state)

            # Backward
            inference_state = predictor.init_state(img_tensor, video_height, video_width)
            _, _, _ = predictor.add_new_points_or_box(
                inference_state=inference_state,
                frame_idx=key_slice, obj_id=1, box=bbox_scaled,
            )
            for out_idx, _, out_logits in predictor.propagate_in_video(
                inference_state, reverse=True
            ):
                mask = (out_logits[0] > 0.0).cpu().numpy()[0]
                segs_3D[out_idx][mask] = 1
            predictor.reset_state(inference_state)

        # Post-processing: largest connected component
        if segs_3D.max() > 0:
            segs_3D = get_largest_cc(segs_3D)

        # Compute Dice
        dice = compute_dice(segs_3D, gts)
        dice_scores.append(dice)

    return np.mean(dice_scores) if dice_scores else 0.0


# ===== TRAINING LOOP =====
print("=" * 60)
print("  MedSAM2 FINE-TUNING -- VENTRICLE SEGMENTATION")
print("=" * 60)
print(f"  Train volumes: {len(train_files)}")
print(f"  Val volumes:   {len(val_files)}")
print(f"  Epochs:        {config['epochs']}")
print(f"  Learning rate: {config['lr']}")
print("=" * 60)

best_dice = 0.0
best_epoch = 0
train_losses = []
val_dices = []
t_start = time.time()

for epoch in range(config['epochs']):
    t_epoch = time.time()

    # Train
    train_loss = train_one_epoch(
        predictor, train_files, config, criterion, optimizer, scaler
    )
    train_losses.append(train_loss)

    # Validate every 5 epochs (or first/last)
    if epoch % 5 == 0 or epoch == config['epochs'] - 1:
        val_dice = validate(predictor, val_files, config)
        val_dices.append((epoch, val_dice))

        # Save best checkpoint
        if val_dice > best_dice:
            best_dice = val_dice
            best_epoch = epoch
            best_path = os.path.join(config['checkpoint_dir'], 'medsam2_ventricle_best.pt')
            torch.save(model.state_dict(), best_path)
            marker = ' ** BEST **'
        else:
            marker = ''

        elapsed = time.time() - t_epoch
        print(f"  Epoch {epoch:3d}/{config['epochs']}  "
              f"loss={train_loss:.4f}  "
              f"val_dice={val_dice:.4f}  "
              f"({elapsed:.1f}s){marker}")
    else:
        elapsed = time.time() - t_epoch
        print(f"  Epoch {epoch:3d}/{config['epochs']}  "
              f"loss={train_loss:.4f}  "
              f"({elapsed:.1f}s)")

total_time = time.time() - t_start

# Save last checkpoint
last_path = os.path.join(config['checkpoint_dir'], 'medsam2_ventricle_last.pt')
torch.save(model.state_dict(), last_path)

print(f"\n{'=' * 60}")
print(f"  TRAINING COMPLETE")
print(f"{'=' * 60}")
print(f"  Total time:      {total_time/60:.1f} minutes")
print(f"  Best Dice:       {best_dice:.4f} (epoch {best_epoch})")
print(f"  Final loss:      {train_losses[-1]:.4f}")
print(f"  Best checkpoint: {best_path}")
print(f"  Last checkpoint: {last_path}")
print(f"{'=' * 60}")

# --- Plot training curves ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(train_losses, color='#ff6e40', linewidth=1.5)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss (Dice + BCE)')
ax1.set_title('Training Loss')
ax1.grid(alpha=0.3)

val_epochs, val_scores = zip(*val_dices)
ax2.plot(val_epochs, val_scores, 'o-', color='#3fb950', linewidth=2, markersize=6)
ax2.axhline(y=best_dice, color='#3fb950', linestyle='--', alpha=0.5,
            label=f'Best: {best_dice:.4f}')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Dice Score')
ax2.set_title('Validation Dice Score')
ax2.set_ylim(0, 1)
ax2.legend()
ax2.grid(alpha=0.3)

plt.suptitle('MedSAM2 Ventricle Fine-Tuning -- Training Curves', fontsize=14)
plt.tight_layout()
plt.show()

<a id="validation-and-metrics"></a>
## 6. Validation and Metrics

Load the best checkpoint and run a full evaluation on the validation set with detailed metrics:
- **Dice Score** -- overlap measure (harmonic mean of precision and recall)
- **IoU (Jaccard)** -- intersection over union
- **Hausdorff Distance (95th percentile)** -- surface distance metric

Visualize predictions vs ground truth for qualitative assessment.

In [ ]:
# =============================================================================
# 6. Validation and Metrics
# =============================================================================
from scipy.spatial import cKDTree
from scipy.ndimage import binary_erosion as sp_binary_erosion


def compute_iou(pred, gt):
    """Compute Intersection over Union (Jaccard index)."""
    pred = pred.astype(bool)
    gt = gt.astype(bool)
    if pred.sum() == 0 and gt.sum() == 0:
        return 1.0
    intersection = np.logical_and(pred, gt).sum()
    union = np.logical_or(pred, gt).sum()
    return float(intersection) / (float(union) + 1e-8)


def compute_hausdorff_95(pred, gt):
    """
    Compute 95th percentile Hausdorff distance between two binary masks.

    Uses surface points (boundary voxels) for computation.
    Returns distance in voxel units.
    """
    pred = pred.astype(bool)
    gt = gt.astype(bool)

    if pred.sum() == 0 or gt.sum() == 0:
        return float('inf') if pred.sum() != gt.sum() else 0.0

    # Extract surface points
    pred_surface = pred ^ sp_binary_erosion(pred)
    gt_surface = gt ^ sp_binary_erosion(gt)

    pred_pts = np.argwhere(pred_surface)
    gt_pts = np.argwhere(gt_surface)

    if len(pred_pts) == 0 or len(gt_pts) == 0:
        return float('inf')

    # Compute distances from pred to gt and gt to pred
    tree_gt = cKDTree(gt_pts)
    dists_p2g, _ = tree_gt.query(pred_pts)

    tree_pred = cKDTree(pred_pts)
    dists_g2p, _ = tree_pred.query(gt_pts)

    all_dists = np.concatenate([dists_p2g, dists_g2p])
    return float(np.percentile(all_dists, 95))


# --- Load best checkpoint ---
best_ckpt_path = os.path.join(config['checkpoint_dir'], 'medsam2_ventricle_best.pt')
print(f"Loading best checkpoint: {best_ckpt_path}")
model.load_state_dict(torch.load(best_ckpt_path, map_location='cuda'))
model.eval()
print(f"Checkpoint loaded (best Dice: {best_dice:.4f} at epoch {best_epoch})")

# --- Run full evaluation on validation set ---
image_size = config['image_size']
img_mean = torch.tensor([0.485, 0.456, 0.406], dtype=torch.float32)[:, None, None].cuda()
img_std = torch.tensor([0.229, 0.224, 0.225], dtype=torch.float32)[:, None, None].cuda()

val_results = []

for npz_path in val_files:
    data = np.load(npz_path)
    imgs = data['imgs']
    gts = data['gts']
    spacing = data['spacing']

    video_height, video_width = imgs.shape[1], imgs.shape[2]

    # Key slice
    slice_areas = gts.sum(axis=(1, 2))
    key_slice = int(np.argmax(slice_areas))
    if slice_areas[key_slice] < config['min_ventricle_area']:
        continue

    bbox = get_bbox_from_mask(gts[key_slice], margin_range=(0.15, 0.15))

    # Preprocess
    img_resized = resize_grayscale_to_rgb(imgs, image_size=image_size)
    img_resized = img_resized / 255.0
    img_tensor = torch.from_numpy(img_resized).cuda().float()
    img_tensor -= img_mean
    img_tensor /= img_std

    scale_x = float(image_size) / video_width
    scale_y = float(image_size) / video_height
    bbox_scaled = np.array([
        bbox[0] * scale_x, bbox[1] * scale_y,
        bbox[2] * scale_x, bbox[3] * scale_y,
    ])

    # Inference
    segs_3D = np.zeros(imgs.shape, dtype=np.uint8)

    with torch.inference_mode(), torch.autocast('cuda', dtype=torch.bfloat16):
        # Forward
        inference_state = predictor.init_state(img_tensor, video_height, video_width)
        _, _, _ = predictor.add_new_points_or_box(
            inference_state=inference_state,
            frame_idx=key_slice, obj_id=1, box=bbox_scaled,
        )
        for out_idx, _, out_logits in predictor.propagate_in_video(inference_state):
            mask = (out_logits[0] > 0.0).cpu().numpy()[0]
            segs_3D[out_idx][mask] = 1
        predictor.reset_state(inference_state)

        # Backward
        inference_state = predictor.init_state(img_tensor, video_height, video_width)
        _, _, _ = predictor.add_new_points_or_box(
            inference_state=inference_state,
            frame_idx=key_slice, obj_id=1, box=bbox_scaled,
        )
        for out_idx, _, out_logits in predictor.propagate_in_video(
            inference_state, reverse=True
        ):
            mask = (out_logits[0] > 0.0).cpu().numpy()[0]
            segs_3D[out_idx][mask] = 1
        predictor.reset_state(inference_state)

    # Post-processing
    if segs_3D.max() > 0:
        segs_3D = get_largest_cc(segs_3D)

    # Metrics
    dice = compute_dice(segs_3D, gts)
    iou = compute_iou(segs_3D, gts)
    hd95 = compute_hausdorff_95(segs_3D, gts)

    val_results.append({
        'file': Path(npz_path).name,
        'dice': dice,
        'iou': iou,
        'hd95': hd95,
        'pred_voxels': int(segs_3D.sum()),
        'gt_voxels': int(gts.sum()),
        'imgs': imgs,
        'gts': gts,
        'preds': segs_3D,
        'key_slice': key_slice,
    })

    print(f"  {Path(npz_path).name}: Dice={dice:.4f}, IoU={iou:.4f}, HD95={hd95:.1f} vx")

# --- Summary ---
print(f"\n{'=' * 60}")
print(f"  VALIDATION METRICS SUMMARY")
print(f"{'=' * 60}")
mean_dice = np.mean([r['dice'] for r in val_results])
mean_iou = np.mean([r['iou'] for r in val_results])
mean_hd95 = np.mean([r['hd95'] for r in val_results if r['hd95'] < float('inf')])
print(f"  Mean Dice:   {mean_dice:.4f}")
print(f"  Mean IoU:    {mean_iou:.4f}")
print(f"  Mean HD95:   {mean_hd95:.1f} voxels")
print(f"  N volumes:   {len(val_results)}")
print(f"{'=' * 60}")

# --- Visualization: predictions vs ground truth ---
for result in val_results:
    imgs_v = result['imgs']
    gts_v = result['gts']
    preds_v = result['preds']
    ks = result['key_slice']

    # Show 4 slices around the key slice
    n_show = 4
    vent_slices = np.where(gts_v.sum(axis=(1, 2)) > 0)[0]
    if len(vent_slices) < n_show:
        show_slices = vent_slices
    else:
        show_indices = np.linspace(0, len(vent_slices) - 1, n_show, dtype=int)
        show_slices = vent_slices[show_indices]

    fig, axes = plt.subplots(3, len(show_slices), figsize=(4 * len(show_slices), 12))
    if len(show_slices) == 1:
        axes = axes.reshape(3, 1)

    for i, z in enumerate(show_slices):
        # Row 0: CT slice
        axes[0, i].imshow(imgs_v[z], cmap='gray')
        axes[0, i].set_title(f'Slice {z}', fontsize=10)
        axes[0, i].axis('off')

        # Row 1: Ground truth overlay
        axes[1, i].imshow(imgs_v[z], cmap='gray')
        gt_rgba = np.zeros((*gts_v[z].shape, 4))
        gt_rgba[:, :, 1] = 1.0  # Green
        gt_rgba[:, :, 3] = gts_v[z].astype(float) * 0.5
        axes[1, i].imshow(gt_rgba)
        axes[1, i].set_title(f'GT ({gts_v[z].sum()} px)', fontsize=10)
        axes[1, i].axis('off')

        # Row 2: Prediction overlay
        axes[2, i].imshow(imgs_v[z], cmap='gray')
        pred_rgba = np.zeros((*preds_v[z].shape, 4))
        pred_rgba[:, :, 0] = 1.0  # Red
        pred_rgba[:, :, 1] = 0.8  # Yellow-ish
        pred_rgba[:, :, 3] = preds_v[z].astype(float) * 0.5
        axes[2, i].imshow(pred_rgba)
        axes[2, i].set_title(f'Pred ({preds_v[z].sum()} px)', fontsize=10)
        axes[2, i].axis('off')

    axes[0, 0].set_ylabel('Brain Window', fontsize=11)
    axes[1, 0].set_ylabel('Ground Truth', fontsize=11)
    axes[2, 0].set_ylabel('Prediction', fontsize=11)

    plt.suptitle(
        f"{result['file']} -- Dice={result['dice']:.4f}, IoU={result['iou']:.4f}",
        fontsize=13
    )
    plt.tight_layout()
    plt.show()

<a id="export-for-deployment"></a>
## 7. Export for Deployment

Save the fine-tuned checkpoint and optionally copy to Google Drive for persistence.

The checkpoint can be:
1. Uploaded to HuggingFace Hub for deployment in a HF Space
2. Used directly in the MedSAM2 inference notebook (`medsam2_ventricle_inference.ipynb`)
3. Integrated with HydroMorph RN via `ApiModelProvider.js` calling a Gradio endpoint

In [ ]:
# =============================================================================
# 7. Export for Deployment
# =============================================================================
import json
from datetime import datetime

# --- Save final fine-tuned checkpoint ---
export_dir = 'checkpoints/finetuned'
os.makedirs(export_dir, exist_ok=True)

best_ckpt = os.path.join(export_dir, 'medsam2_ventricle_best.pt')
last_ckpt = os.path.join(export_dir, 'medsam2_ventricle_last.pt')

print(f"Fine-tuned checkpoints:")
print(f"  Best: {best_ckpt} ({os.path.getsize(best_ckpt) / 1e6:.1f} MB)")
print(f"  Last: {last_ckpt} ({os.path.getsize(last_ckpt) / 1e6:.1f} MB)")

# --- Save training metadata ---
metadata = {
    'timestamp': datetime.now().isoformat(),
    'model': 'MedSAM2 (fine-tuned for ventricle segmentation)',
    'base_checkpoint': 'MedSAM2_2411.pt',
    'config': {
        k: v for k, v in config.items()
        if not isinstance(v, (type, torch.Tensor))
    },
    'training': {
        'train_volumes': len(train_files),
        'val_volumes': len(val_files),
        'best_epoch': int(best_epoch),
        'best_dice': float(best_dice),
        'final_loss': float(train_losses[-1]) if train_losses else None,
        'total_time_min': float(total_time / 60),
    },
    'validation': {
        'mean_dice': float(mean_dice),
        'mean_iou': float(mean_iou),
        'mean_hd95': float(mean_hd95) if mean_hd95 < float('inf') else None,
    },
    'frozen_layers': 'image_encoder',
    'trainable_params': trainable_params,
    'total_params': total_params,
    'data_note': 'Trained on synthetic data. Real ventricle-labeled CT data required for production.',
}

metadata_path = os.path.join(export_dir, 'training_metadata.json')
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)
print(f"\nTraining metadata saved: {metadata_path}")

# --- Optional: Save to Google Drive ---
# Uncomment to persist across Colab sessions:
# from google.colab import drive
# drive.mount('/content/drive')
#
# DRIVE_DIR = '/content/drive/MyDrive/hydromorph-medsam2'
# os.makedirs(DRIVE_DIR, exist_ok=True)
# shutil.copy2(best_ckpt, f'{DRIVE_DIR}/medsam2_ventricle_best.pt')
# shutil.copy2(metadata_path, f'{DRIVE_DIR}/training_metadata.json')
# print(f"Saved to Google Drive: {DRIVE_DIR}/")

print(f"\n--- Deployment Instructions ---")
print(f"")
print(f"1. Upload checkpoint to HuggingFace:")
print(f"   huggingface-cli upload mmrech/medsam2-ventricle {best_ckpt}")
print(f"")
print(f"2. Use in MedSAM2 inference notebook:")
print(f"   Replace checkpoint path with fine-tuned version")
print(f"")
print(f"3. Deploy to HuggingFace Space:")
print(f"   Create Gradio app with the fine-tuned model")
print(f"   HydroMorph RN calls via GradioClient.js or ApiModelProvider.js")

<a id="comparison"></a>
## 8. Comparison: Base vs Fine-Tuned Model

Run inference with both the **base MedSAM2** checkpoint and the **fine-tuned** checkpoint on the same validation volume. This quantifies the improvement from fine-tuning on ventricle-specific data.

In [ ]:
# =============================================================================
# 8. Comparison: Base vs Fine-Tuned
# =============================================================================


@torch.inference_mode()
def run_inference(predictor, imgs, gts, config):
    """
    Run full MedSAM2 inference on a single volume.

    Returns the predicted 3D binary mask.
    """
    image_size = config['image_size']
    video_height, video_width = imgs.shape[1], imgs.shape[2]

    img_mean = torch.tensor([0.485, 0.456, 0.406], dtype=torch.float32)[:, None, None].cuda()
    img_std = torch.tensor([0.229, 0.224, 0.225], dtype=torch.float32)[:, None, None].cuda()

    # Find key slice
    slice_areas = gts.sum(axis=(1, 2))
    key_slice = int(np.argmax(slice_areas))
    bbox = get_bbox_from_mask(gts[key_slice], margin_range=(0.15, 0.15))

    # Preprocess
    img_resized = resize_grayscale_to_rgb(imgs, image_size=image_size)
    img_resized = img_resized / 255.0
    img_tensor = torch.from_numpy(img_resized).cuda().float()
    img_tensor -= img_mean
    img_tensor /= img_std

    scale_x = float(image_size) / video_width
    scale_y = float(image_size) / video_height
    bbox_scaled = np.array([
        bbox[0] * scale_x, bbox[1] * scale_y,
        bbox[2] * scale_x, bbox[3] * scale_y,
    ])

    segs = np.zeros(imgs.shape, dtype=np.uint8)

    with torch.autocast('cuda', dtype=torch.bfloat16):
        # Forward
        state = predictor.init_state(img_tensor, video_height, video_width)
        _, _, _ = predictor.add_new_points_or_box(
            inference_state=state, frame_idx=key_slice, obj_id=1, box=bbox_scaled
        )
        for idx, _, logits in predictor.propagate_in_video(state):
            segs[idx][(logits[0] > 0.0).cpu().numpy()[0]] = 1
        predictor.reset_state(state)

        # Backward
        state = predictor.init_state(img_tensor, video_height, video_width)
        _, _, _ = predictor.add_new_points_or_box(
            inference_state=state, frame_idx=key_slice, obj_id=1, box=bbox_scaled
        )
        for idx, _, logits in predictor.propagate_in_video(state, reverse=True):
            segs[idx][(logits[0] > 0.0).cpu().numpy()[0]] = 1
        predictor.reset_state(state)

    if segs.max() > 0:
        segs = get_largest_cc(segs)

    return segs


# Pick a validation volume for comparison
comp_data = np.load(val_files[0])
comp_imgs = comp_data['imgs']
comp_gts = comp_data['gts']
comp_spacing = comp_data['spacing']

print(f"Comparison volume: {Path(val_files[0]).name}")
print(f"  Shape: {comp_imgs.shape}")
print(f"  GT ventricle voxels: {comp_gts.sum()}")

# --- Run inference with BASE model ---
print(f"\nLoading base MedSAM2 checkpoint...")
base_state = torch.load(checkpoint, map_location='cuda')
model.load_state_dict(base_state, strict=False)
model.eval()

print("Running base model inference...")
t0 = time.time()
base_pred = run_inference(predictor, comp_imgs, comp_gts, config)
base_time = time.time() - t0
base_dice = compute_dice(base_pred, comp_gts)
base_iou = compute_iou(base_pred, comp_gts)
print(f"  Base: Dice={base_dice:.4f}, IoU={base_iou:.4f} ({base_time:.1f}s)")

# --- Run inference with FINE-TUNED model ---
print(f"\nLoading fine-tuned checkpoint...")
ft_state = torch.load(best_ckpt, map_location='cuda')
model.load_state_dict(ft_state)
model.eval()

print("Running fine-tuned model inference...")
t0 = time.time()
ft_pred = run_inference(predictor, comp_imgs, comp_gts, config)
ft_time = time.time() - t0
ft_dice = compute_dice(ft_pred, comp_gts)
ft_iou = compute_iou(ft_pred, comp_gts)
print(f"  Fine-tuned: Dice={ft_dice:.4f}, IoU={ft_iou:.4f} ({ft_time:.1f}s)")

# --- Comparison summary ---
print(f"\n{'=' * 60}")
print(f"  BASE vs FINE-TUNED COMPARISON")
print(f"{'=' * 60}")
print(f"  {'Metric':<15} {'Base':>10} {'Fine-tuned':>12} {'Delta':>10}")
print(f"  {'-'*47}")
print(f"  {'Dice':<15} {base_dice:>10.4f} {ft_dice:>12.4f} {ft_dice-base_dice:>+10.4f}")
print(f"  {'IoU':<15} {base_iou:>10.4f} {ft_iou:>12.4f} {ft_iou-base_iou:>+10.4f}")
print(f"  {'Voxels':<15} {base_pred.sum():>10,} {ft_pred.sum():>12,} {ft_pred.sum()-base_pred.sum():>+10,}")
print(f"  {'GT Voxels':<15} {comp_gts.sum():>10,}")
print(f"{'=' * 60}")

if ft_dice > base_dice:
    pct_improvement = (ft_dice - base_dice) / max(base_dice, 1e-8) * 100
    print(f"\n  Fine-tuning improved Dice by {pct_improvement:.1f}%")
else:
    print(f"\n  Note: Fine-tuning on synthetic data may not improve over base model.")
    print(f"  Real ventricle-labeled CT data is needed for meaningful improvement.")

# --- Side-by-side visualization ---
vent_slices = np.where(comp_gts.sum(axis=(1, 2)) > 0)[0]
n_show = min(4, len(vent_slices))
show_indices = np.linspace(0, len(vent_slices) - 1, n_show, dtype=int)
show_slices = vent_slices[show_indices]

fig, axes = plt.subplots(4, n_show, figsize=(4 * n_show, 16))
if n_show == 1:
    axes = axes.reshape(4, 1)

for i, z in enumerate(show_slices):
    # Row 0: CT
    axes[0, i].imshow(comp_imgs[z], cmap='gray')
    axes[0, i].set_title(f'Slice {z}', fontsize=10)
    axes[0, i].axis('off')

    # Row 1: Ground truth
    axes[1, i].imshow(comp_imgs[z], cmap='gray')
    gt_rgba = np.zeros((*comp_gts[z].shape, 4))
    gt_rgba[:, :, 1] = 1.0
    gt_rgba[:, :, 3] = comp_gts[z].astype(float) * 0.5
    axes[1, i].imshow(gt_rgba)
    axes[1, i].set_title(f'GT', fontsize=10)
    axes[1, i].axis('off')

    # Row 2: Base model
    axes[2, i].imshow(comp_imgs[z], cmap='gray')
    base_rgba = np.zeros((*base_pred[z].shape, 4))
    base_rgba[:, :, 0] = 0.25
    base_rgba[:, :, 1] = 0.73
    base_rgba[:, :, 2] = 0.31
    base_rgba[:, :, 3] = base_pred[z].astype(float) * 0.5
    axes[2, i].imshow(base_rgba)
    axes[2, i].set_title(f'Base ({base_pred[z].sum()} px)', fontsize=10)
    axes[2, i].axis('off')

    # Row 3: Fine-tuned model
    axes[3, i].imshow(comp_imgs[z], cmap='gray')
    ft_rgba = np.zeros((*ft_pred[z].shape, 4))
    ft_rgba[:, :, 0] = 1.0
    ft_rgba[:, :, 1] = 0.8
    ft_rgba[:, :, 3] = ft_pred[z].astype(float) * 0.5
    axes[3, i].imshow(ft_rgba)
    axes[3, i].set_title(f'Fine-tuned ({ft_pred[z].sum()} px)', fontsize=10)
    axes[3, i].axis('off')

axes[0, 0].set_ylabel('Brain Window', fontsize=11)
axes[1, 0].set_ylabel('Ground Truth', fontsize=11)
axes[2, 0].set_ylabel(f'Base (Dice={base_dice:.3f})', fontsize=11)
axes[3, 0].set_ylabel(f'Fine-tuned (Dice={ft_dice:.3f})', fontsize=11)

plt.suptitle('Base MedSAM2 vs Fine-Tuned -- Ventricle Segmentation', fontsize=14)
plt.tight_layout()
plt.show()

<a id="next-steps"></a>
## 9. Next Steps

### More Training Data

This notebook used **synthetic data** to demonstrate the fine-tuning pipeline. For production-quality ventricle segmentation, real training data is essential:

- **[TotalSegmentator](https://github.com/wasserth/TotalSegmentator):** Run on head CT volumes to generate automatic ventricle masks (class: `brain_ventricle`). 50-100 volumes recommended.
- **Institutional data:** Head CT scans with manual ventricle annotations by neuroradiologists. Requires IRB approval.
- **[Medical Segmentation Decathlon](http://medicaldecathlon.com/):** Brain tumor data (Task01_BrainTumour) -- MRI, not CT, but may provide transfer learning benefit.

Use the `nifti_to_npz()` function from Cell 3 to convert any NIfTI CT + mask pairs to MedSAM2 training format.

### Multi-Class Fine-Tuning

Extend beyond binary ventricle segmentation to capture other NPH-relevant features:
- Lateral ventricles (left + right separately)
- Third ventricle
- Fourth ventricle
- Periventricular hyperintensity / transependymal edema
- Sylvian fissure dilation

### Deploy to HuggingFace Space

1. Create a Gradio app wrapping the fine-tuned MedSAM2 model
2. Deploy to a **ZeroGPU** HuggingFace Space (free tier: ~300s/day GPU)
3. Expose as API endpoint for HydroMorph RN app

### Integration with HydroMorph RN

Update the model registry (`src/models/ModelRegistry.js`) to point the `medsam2` slot to the deployed HuggingFace Space:

```javascript
{
  id: 'medsam2',
  name: 'MedSAM2',
  provider: 'api',
  endpoint: 'https://mmrech-medsam2-ventricle.hf.space',
  fallbackToMock: true,  // Keep mock fallback for offline use
}
```

Create `ApiModelProvider.js` implementing the same interface as `MockModelProvider.js` to send NIfTI data to the Gradio endpoint and receive the segmentation mask.

### Efficient MedSAM2 Variant

For CPU-only deployment, consider fine-tuning the **Efficient MedSAM2 Tiny** variant (`eff_medsam2_tiny_FLARE25_RECIST_baseline.pt`, ~72 MB). It runs on CPU at ~15 seconds per volume, enabling deployment on free HuggingFace CPU Spaces.

### References

- [MedSAM2 GitHub](https://github.com/bowang-lab/MedSAM2)
- [MedSAM2 Paper](https://arxiv.org/abs/2408.00874) -- Zhu et al., 2024
- [SAM2 Paper (Meta)](https://arxiv.org/abs/2408.00714) -- Ravi et al., 2024
- [Evans Index -- Radiopaedia](https://radiopaedia.org/articles/evans-index)
- [TotalSegmentator](https://github.com/wasserth/TotalSegmentator)
- [HydroMorph RN App](https://github.com/matheusrech/hydromorph-rn)
- [MONAI Brain Segmentation](https://monai.io/) -- see `monai_brain_segmentation.ipynb`